## **Installs**

In [ ]:
#!pip install -U transformers

## **Hugging Face Token**

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("my_hf_token")

In [ ]:
from huggingface_hub import login
login(token=hf_token)

## **Imports**

In [ ]:
import json
import re
import math
import random
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import gc, traceback

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
MODEL_NAME = "google/gemma-2b"
HF_TOKEN = hf_token

## **Load Model**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,      # NOT torch_dtype (deprecated, as you already found)
    device_map="auto",
)
model.eval()

DEVICE = next(model.parameters()).device
NUM_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
print(f"Loaded {MODEL_NAME}: {NUM_LAYERS} layers, hidden_dim={HIDDEN_DIM}, device={DEVICE}")

In [ ]:
#freeze model params

for p in model.parameters():
    p.requires_grad_(False)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params in base model (should be 0): {trainable}")

## **Dataset Loading**

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/anarvaaa/dataset-for-ssv/pico_dataset.csv")  # update path if running locally
# If running outside Kaggle, point this at your local copy instead, e.g.:
# df = pd.read_csv("pico_dataset.csv")
print(df.shape)
df.head(3)

## **Generate negative (unsteered) responses**

For each question, we run the base model with *just the question* -- no PICO
instruction, no steering -- and let it continue naturally. Whatever it produces
becomes the `negative_response`: an example of what the model says when it is
**not** pushed toward PICO format. The existing `P`/`I`/`C`/`O` ground truth stays
as the positive (steered) example for each row. This gives us the
`(x, y, y^c)` preference triples the RePS/BiPO-style objectives need for
experiment 2 -- unlike experiment 1, which only used `(x, y^c)` pairs with a
plain language-modeling loss.

In [ ]:
GEN_MAX_NEW_TOKENS = 100
GEN_DO_SAMPLE = True
GEN_TEMPERATURE = 1.0
GEN_TOP_P = 0.95

@torch.no_grad()
def generate_negative(question, max_new_tokens=GEN_MAX_NEW_TOKENS):
    prompt = f"{question.strip()}\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=GEN_DO_SAMPLE,
        temperature=GEN_TEMPERATURE,
        top_p=GEN_TOP_P,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return text.strip()

In [ ]:
negative_responses = []

for i, row in df.iterrows():
    neg = generate_negative(row["research_question"])
    negative_responses.append(neg)
    if (i + 1) % 5 == 0 or (i + 1) == len(df):
        print(f"[{i+1}/{len(df)}] done")
        print("  Q:", row["research_question"][:100])
        print("  neg:", neg[:150].replace("\n", " "))

In [ ]:
df_out = df.copy()
df_out["negative_response"] = negative_responses

out_path = "/kaggle/working/pico_dataset_with_negatives.csv"
df_out.to_csv(out_path, index=False)
print(f"Saved {df_out.shape} to {out_path}")
df_out.head(3)